In [ ]:
# Montar google drive y verificar que los CSV existen y son alcanzables
from google.colab import drive
drive.mount('/content/drive')

import os

base_path = "/content/drive/MyDrive/TFG_Data/PROCESSED"

# Instalar dependencias
!pip install -q transformers datasets accelerate torch scikit-learn sentencepiece

# Cargar los dataset
import pandas as pd

def load_csv(name):
    path = f"/content/drive/MyDrive/TFG_Data/PROCESSED/{name}"
    df = pd.read_csv(path)
    df = df.dropna()
    return df['text'].tolist(), df['label'].tolist()

# Para la implementación inicial se emplearán los de IMBD
train_texts, train_labels = load_csv("IMDB_train.csv")
test_texts,  test_labels  = load_csv("IMDB_test.csv")

len(train_texts), len(test_texts)

# Smoke test de BERT
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

def test_bert(model_name, test_texts, test_labels):
    print("Cargando BERT:", model_name)
    # Se usa device_map="auto" para deficits de hardware
    classifier = pipeline("sentiment-analysis", model=model_name, truncation=True, device_map="auto")

    preds = []

    for text in test_texts[:500]:
        out = classifier(text)[0]["label"]
        num = int(out[0])
        label = 1 if num >= 4 else 0
        preds.append(label)

    acc = accuracy_score(test_labels[:500], preds)
    f1  = f1_score(test_labels[:500], preds)
    return acc, f1

# Smoke test de GPT
from transformers import pipeline

def test_gpt(model_name, test_texts, test_labels):
    print("Cargando GPT:", model_name)
    gpt = pipeline("text-generation", model=model_name, device_map="auto")

    if gpt.tokenizer.pad_token_id is None:
        gpt.tokenizer.pad_token_id = gpt.tokenizer.eos_token_id

    preds = []
    max_new_tokens_gpt = 10

    # Se define max_length por el tamaño generado de los tokens
    model_max_input_length = gpt.tokenizer.model_max_length - max_new_tokens_gpt

    for text in test_texts[:200]:

        raw_prompt_string = (
            "Clasificación del sentimiento como Positivo o Negativo.\n"
            f"Reseña: {text}\n"
            "Sentimiento:"
        )

        # se tokeniza y trunca el input para asegurar que entre
        inputs = gpt.tokenizer(
            raw_prompt_string,
            return_tensors="pt",
            truncation=True,
            max_length=model_max_input_length
        ).to(gpt.device)

        # Se genera la salida
        output = gpt.model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens_gpt,
            do_sample=False,
            pad_token_id=gpt.tokenizer.pad_token_id
        )

        # Se decodifica solo los nuevos tokens generados
        decoded_output = gpt.tokenizer.decode(
            output[0, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).lower()

        label = 1 if "pos" in decoded_output else 0
        preds.append(label)

    acc = accuracy_score(test_labels[:200], preds)
    f1  = f1_score(test_labels[:200], preds)
    return acc, f1

# Smoke test de LlaMA (TinyLlaMA porque es mucho más ligero)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

def test_llama(model_name, test_texts, test_labels):
    print("Cargando LLaMA:", model_name)

    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

    preds = []
    for text in test_texts[:200]:

        prompt = (
            "Clasificación del sentimiento como Positivo o Negativo.\n"
            f"Reseña: {text}\n"
            "Sentimiento:"
        )

        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=model.config.max_position_embeddings)
        out = model.generate(**inputs, max_new_tokens=12)
        decoded = tok.decode(out[0]).lower()

        label = 1 if "pos" in decoded else 0
        preds.append(label)

    acc = accuracy_score(test_labels[:200], preds)
    f1  = f1_score(test_labels[:200], preds)
    return acc, f1

# Ejecución de las smoke test
bert_acc, bert_f1 = test_bert(
    "nlptown/bert-base-multilingual-uncased-sentiment",
    test_texts, test_labels
)

gpt_acc, gpt_f1 = test_gpt(
    "gpt2",
    test_texts, test_labels
)

llama_acc, llama_f1 = test_llama(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    test_texts, test_labels
)

# Resultados
print("\n===== RESULTADOS DEL SMOKE TEST =====\n")
print(f"BERT   → ACC: {bert_acc:.4f}   F1: {bert_f1:.4f}")
print(f"GPT    → ACC: {gpt_acc:.4f}    F1: {gpt_f1:.4f}")
print(f"LLaMA  → ACC: {llama_acc:.4f}  F1: {llama_f1:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando BERT: nlptown/bert-base-multilingual-uncased-sentiment


Device set to use cpu


Cargando GPT: gpt2


Device set to use cpu


Cargando LLaMA: TinyLlama/TinyLlama-1.1B-Chat-v1.0

===== RESULTADOS DEL SMOKE TEST =====

BERT   → ACC: 0.8540   F1: 0.8396
GPT    → ACC: 0.4850    F1: 0.0190
LLaMA  → ACC: 0.5200  F1: 0.6842
